In [1]:
!pip install -q -U transformers accelerate


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.6/12.6 MB 107.2 MB/s eta 0:00:0000:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 394.3/394.3 kB 19.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 846.4/846.4 kB 23.4 MB/s eta 0:00:0000:01


In [2]:
pip install num2words

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 163.5/163.5 kB 7.2 MB/s eta 0:00:00
  Created wheel for docopt: filename=docopt-0.6.2-py2.py3-none-any.whl size=13781 sha256=197a3743d66cd7b6a8db8ea756fc8d68a4c121356ef65c74b24392dbdac0adea
  Stored in directory: /root/.cache/pip/wheels/0b/1d/03/175286677fb5a1341cc3e4755bf8ec0ed08f3329afd67446b0
Successfully built docopt
Note: you may need to restart the kernel to use updated packages.


In [3]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
import base64
import os
import json
import os
from io import BytesIO
from PIL import Image

In [4]:
import torch

print(torch.cuda.is_available())
print(torch.cuda.get_device_name(0) if torch.cuda.is_available() else "GPU unavailable")

True
Tesla T4


In [5]:
import torch
from transformers import AutoProcessor, AutoModelForImageTextToText

MODEL_NAME = "HuggingFaceTB/SmolVLM2-2.2B-Instruct"

processor_smol= AutoProcessor.from_pretrained(MODEL_NAME)

model_smol = AutoModelForImageTextToText.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float16,
    low_cpu_mem_usage=True
).to("cuda")

model_smol.eval()

print("SmolVLM2 loaded successfully!")

processor_config.json:   0%|          | 0.00/67.0 [00:00<?, ?B/s]

chat_template.json:   0%|          | 0.00/430 [00:00<?, ?B/s]

preprocessor_config.json:   0%|          | 0.00/599 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/3.64k [00:00<?, ?B/s]

[transformers] Model config: pad_token_id must be `None` or an integer within the vocabulary (between 0 and 31999), got 128002. This may result in unexpected behavior.


...

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/657 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/136 [00:00<?, ?B/s]

SmolVLM2 loaded successfully!


In [6]:
def query_from_path(image_path):
    # Helper to look up the correct query based on the image_path file name
    target_name = "images/" + os.path.basename(image_path)
    for record in annotations_list:
        if record["image_path"] == target_name:
            return record["question"]
    raise ValueError(f"No annotation record found for image path: {image_path}")

In [7]:
def get_base64_from_rgb(rgb_image):
    # Helper to convert in-memory numpy array (RGB) to base64 string
    bgr_img = cv2.cvtColor(rgb_image, cv2.COLOR_RGB2BGR)
    _, buffer = cv2.imencode('.jpg', bgr_img)
    return base64.b64encode(buffer).decode('utf-8')

In [8]:
def evaluate_original(image_path):
  with open(image_path, "rb") as f:
    base64_image = base64.b64encode(f.read()).decode("utf-8")
  query_text = query_from_path(image_path)
  # Predict
  result = evaluate(query_text,base64_image)
  return result

In [9]:

def evaluate(query_text, b64_str):
    image_bytes = base64.b64decode(b64_str)
    image = Image.open(BytesIO(image_bytes)).convert("RGB")

    question = query_text + "\nAnswer only Yes or No."

    messages = [
        {
            "role": "user",
            "content": [
                {"type": "image"},
                {"type": "text", "text": question}
            ]
        }
    ]

    text = processor_smol.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    inputs = processor_smol(
    text=[text],
    images=[image],
    padding=True,
    return_tensors="pt"
    ).to(model_smol.device)

    with torch.inference_mode():
        output_ids = model_smol.generate(
            **inputs,
            max_new_tokens=10,
            do_sample=False
        )

    generated_ids = output_ids[:, len(inputs["input_ids"][0]):]

    answer = processor_smol.batch_decode(
        generated_ids,
        skip_special_tokens=True
    )[0]

    return answer.strip()



In [10]:
import transformers
print(transformers.__version__)

5.19.0


In [11]:
import os
import pandas as pd
from tqdm.notebook import tqdm

# --- Paths Configuration (Adjust as needed) ---
IMAGE_DIR = "/kaggle/input/datasets/ramakanthjogada/true-positive-only/repope_yes_500_unique/images"
ANNOTATIONS_PATH = "/kaggle/input/datasets/ramakanthjogada/true-positive-only/repope_yes_500_unique/annotations.json"

# Load annotations lookup map for faster access
with open(ANNOTATIONS_PATH, "r") as f:
    annotations_list = json.load(f)

# Create a lookup map for ground truth: (image_id) -> label
# Since we are evaluating the actual files in /images/, we target '/images/' records
ground_truth_map = {}
for record in annotations_list:
    if record["image_path"].startswith("images/"):
        img_id = record["image_id"]
        ground_truth_map[img_id] = record["label"]

# Get list of all images in the directory
image_files = [f for f in os.listdir(IMAGE_DIR) if f.endswith(('.jpg', '.jpeg', '.png'))]

results = []

print(f"Processing {len(image_files)} images through all variations...")
for filename in tqdm(image_files):
    # Extract image_id from filename (e.g., '000000046207.jpg' -> 46207)
    try:
        image_id = int(os.path.splitext(filename)[0])
    except ValueError:
        continue

    image_path = os.path.join(IMAGE_DIR, filename)
    ground_truth = ground_truth_map.get(image_id, "Unknown")

    # Evaluate under all conditions
    try:
        orig_val = evaluate_original(image_path)
    except Exception as e:
        orig_val = f"Error: {str(e)}"
         
        
        
    results.append({
        "Image_Id": image_id,
        "Ground_truth": ground_truth,
        "Original": orig_val})

df_results = pd.DataFrame(results)
display(df_results.head(10))

Processing 491 images through all variations...


  0%|          | 0/491 [00:00<?, ?it/s]

[transformers] Kwargs passed to `processor.__call__` have to be in `processor_kwargs` dict, not in `**kwargs`
[transformers] Kwargs passed to `processor.__call__` have to be in `processor_kwargs` dict, not in `**kwargs`
[transformers] Kwargs passed to `processor.__call__` have to be in `processor_kwargs` dict, not in `**kwargs`
[transformers] Kwargs passed to `processor.__call__` have to be in `processor_kwargs` dict, not in `**kwargs`
[transformers] Kwargs passed to `processor.__call__` have to be in `processor_kwargs` dict, not in `**kwargs`
[transformers] Kwargs passed to `processor.__call__` have to be in `processor_kwargs` dict, not in `**kwargs`
[transformers] Kwargs passed to `processor.__call__` have to be in `processor_kwargs` dict, not in `**kwargs`
[transformers] Kwargs passed to `processor.__call__` have to be in `processor_kwargs` dict, not in `**kwargs`
[transformers] Kwargs passed to `processor.__call__` have to be in `processor_kwargs` dict, not in `**kwargs`
[transform

,Image_Id,Ground_truth,Original
0,102439,Unknown,Yes.
1,236865,Unknown,Yes.
2,327038,Unknown,Yes.
3,381895,Unknown,Yes.
4,108189,Unknown,Yes.
5,153865,Unknown,Yes.
6,560744,Unknown,Yes.
7,408757,Unknown,Yes.
8,562155,Unknown,Yes.
9,579231,Unknown,No.


In [14]:
df_results["Ground_truth"] = df_results["Ground_truth"].replace("Unknown", "yes") # As I took all ground_truth as Yes 

In [15]:
df_results

,Image_Id,Ground_truth,Original
0,102439,yes,Yes.
1,236865,yes,Yes.
2,327038,yes,Yes.
3,381895,yes,Yes.
4,108189,yes,Yes.
...,...,...,...
486,172648,yes,Yes.
487,365317,yes,Yes.
488,573796,yes,Yes.
489,528136,yes,Yes.


In [17]:
df_results.rename(columns={"Original": "Smol-2.2B"}, inplace=True)

In [18]:
df_results

,Image_Id,Ground_truth,Smol-2.2B
0,102439,yes,Yes.
1,236865,yes,Yes.
2,327038,yes,Yes.
3,381895,yes,Yes.
4,108189,yes,Yes.
...,...,...,...
486,172648,yes,Yes.
487,365317,yes,Yes.
488,573796,yes,Yes.
489,528136,yes,Yes.


In [21]:
import pandas as pd

df = pd.read_csv("/kaggle/input/datasets/ramakanthjogada/diff-models-evals-on-pope/diff_models_eval (1).csv", usecols=["Image_Id","Ground_truth","Smol-2.2B"])
df

,Image_Id,Ground_truth,Smol-2.2B
0,6578,No,Yes.
1,8602,No,No.
2,9469,No,No.
3,9712,No,No.
4,10733,No,No.
...,...,...,...
495,574356,No,No.
496,574837,No,No.
497,575121,No,No.
498,575916,No,No.


In [22]:
append_df = pd.concat([df,df_results], ignore_index=True)

In [23]:
append_df

,Image_Id,Ground_truth,Smol-2.2B
0,6578,No,Yes.
1,8602,No,No.
2,9469,No,No.
3,9712,No,No.
4,10733,No,No.
...,...,...,...
986,172648,yes,Yes.
987,365317,yes,Yes.
988,573796,yes,Yes.
989,528136,yes,Yes.


In [24]:
df1 = pd.read_csv("/kaggle/input/datasets/ramakanthjogada/final-degradation/final-degradation.csv", usecols=["Image_Id","Ground_truth","Original"])
df1

,Image_Id,Ground_truth,Original
0,169361,No,No
1,47022,No,No
2,556064,No,No
3,474004,No,No
4,306710,No,No
...,...,...,...
986,497599,yes,Yes.
987,121959,yes,Yes
988,361430,yes,No
989,277289,yes,Yes


In [25]:
df1.rename(columns={"Original": "Qwen-3B"}, inplace=True)

In [26]:
df1

,Image_Id,Ground_truth,Qwen-3B
0,169361,No,No
1,47022,No,No
2,556064,No,No
3,474004,No,No
4,306710,No,No
...,...,...,...
986,497599,yes,Yes.
987,121959,yes,Yes
988,361430,yes,No
989,277289,yes,Yes


In [36]:
print("Duplicates in df1:", df1["Image_Id"].duplicated().sum())
print("Duplicates in append_df:", append_df["Image_Id"].duplicated().sum())

Duplicates in df1: 2
Duplicates in append_df: 2


In [37]:
duplicates = df1[df1["Image_Id"].duplicated(keep=False)]

print(duplicates[["Image_Id"]])
print("Indices:", duplicates.index.tolist())  

     Image_Id
229    156704
341     44801
823     44801
850    156704
Indices: [229, 341, 823, 850]


In [39]:
df1 = df1.drop_duplicates(subset=["Image_Id"], keep="first")
append_df = append_df.drop_duplicates(subset=["Image_Id"], keep="first")

merged_df = pd.merge(df1, append_df, on="Image_Id", how="inner")
merged_df.reset_index(drop=True, inplace=True)

In [40]:
merged_df

,Image_Id,Ground_truth_x,Qwen-3B,Ground_truth_y,Smol-2.2B
0,169361,No,No,No,No.
1,47022,No,No,No,Yes.
2,556064,No,No,No,No.
3,474004,No,No,No,Yes.
4,306710,No,No,No,Yes.
...,...,...,...,...,...
984,497599,yes,Yes.,yes,Yes.
985,121959,yes,Yes,yes,Yes.
986,361430,yes,No,yes,No.
987,277289,yes,Yes,yes,Yes.


In [42]:
merged_df.drop(columns=["Ground_truth_y"], inplace=True)
merged_df

,Image_Id,Ground_truth_x,Qwen-3B,Smol-2.2B
0,169361,No,No,No.
1,47022,No,No,Yes.
2,556064,No,No,No.
3,474004,No,No,Yes.
4,306710,No,No,Yes.
...,...,...,...,...
984,497599,yes,Yes.,Yes.
985,121959,yes,Yes,Yes.
986,361430,yes,No,No.
987,277289,yes,Yes,Yes.


In [45]:
merged_df.rename(columns={"Ground_truth_x": "Ground_truth"}, inplace=True)

In [46]:
merged_df

,Image_Id,Ground_truth,Qwen-3B,Smol-2.2B
0,169361,No,No,No.
1,47022,No,No,Yes.
2,556064,No,No,No.
3,474004,No,No,Yes.
4,306710,No,No,Yes.
...,...,...,...,...
984,497599,yes,Yes.,Yes.
985,121959,yes,Yes,Yes.
986,361430,yes,No,No.
987,277289,yes,Yes,Yes.


In [47]:
merged_df.to_csv("two-model.csv",index=False)